# Movie Review Sentiment Analysis Using RNN - PyTorch Version
This notebook contains the PyTorch implementation of an RNN model for sentiment classification.
The model classifies movie reviews into 5 sentiment categories: Very Negative (0), Negative (1), Neutral (2), Positive (3), Very Positive (4)

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pickle

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load the dataset
train = pd.read_csv('train.tsv', sep='\t')
print(f"Dataset shape: {train.shape}")
print(f"\nFirst few samples:")
print(train.head(10))

In [ ]:
# Check dataset info
print(f"Dataset shape: {train.shape}")
print(f"\nSentiment distribution:")
print(train['Sentiment'].value_counts())
print(f"\nNumber of sentiments: {train['Sentiment'].nunique()}")

In [ ]:
# Extract features and labels
X = train.loc[:, 'Phrase']
y = train.loc[:, 'Sentiment']

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

In [ ]:
# Create a Tokenizer class to handle text preprocessing
class SimpleTokenizer:
    def __init__(self, oov_token='<nothing>'):
        self.word_index = {}
        self.word_counts = {}
        self.document_count = 0
        self.oov_token = oov_token
    
    def fit_on_texts(self, texts):
        """Build vocabulary from texts"""
        self.word_index = {self.oov_token: 1}  # Reserve index 0 for padding
        index = 2
        
        for text in texts:
            words = text.lower().split()
            for word in words:
                if word not in self.word_index:
                    self.word_index[word] = index
                    self.word_counts[word] = 1
                    index += 1
                else:
                    self.word_counts[word] = self.word_counts.get(word, 0) + 1
        
        self.document_count = len(texts)
    
    def texts_to_sequences(self, texts):
        """Convert texts to sequences of integers"""
        sequences = []
        for text in texts:
            words = text.lower().split()
            seq = [self.word_index.get(word, self.word_index[self.oov_token]) for word in words]
            sequences.append(seq)
        return sequences


# Create tokenizer and fit on training data
tokenizer = SimpleTokenizer(oov_token='<nothing>')
tokenizer.fit_on_texts(X_train)

print(f"Vocabulary size: {len(tokenizer.word_index)}")
print(f"Total documents: {tokenizer.document_count}")
print(f"Sample word_index entries: {dict(list(tokenizer.word_index.items())[:10])}")

In [ ]:
# Convert texts to sequences
train_seq = tokenizer.texts_to_sequences(X_train)
test_seq = tokenizer.texts_to_sequences(X_test)

print(f"Number of training sequences: {len(train_seq)}")
print(f"Number of test sequences: {len(test_seq)}")
print(f"\nSample training sequence: {train_seq[0]}")

In [ ]:
# Pad sequences to same length
def pad_sequences_custom(sequences, maxlen=None, padding='post', pad_value=0):
    """Pad sequences to same length"""
    if maxlen is None:
        maxlen = max([len(seq) for seq in sequences]) if sequences else 0
    
    padded = []
    for seq in sequences:
        if padding == 'post':
            padded_seq = seq + [pad_value] * (maxlen - len(seq))
        else:  # 'pre'
            padded_seq = [pad_value] * (maxlen - len(seq)) + seq
        padded.append(padded_seq)
    
    return np.array(padded), maxlen


train_seq_padded, max_len = pad_sequences_custom(train_seq, padding='post')
test_seq_padded, _ = pad_sequences_custom(test_seq, padding='post', maxlen=max_len)

print(f"Padded train sequences shape: {train_seq_padded.shape}")
print(f"Padded test sequences shape: {test_seq_padded.shape}")
print(f"Maximum sequence length: {max_len}")

In [ ]:
# Convert labels to one-hot encoding
def one_hot_encode(labels, num_classes):
    """Convert labels to one-hot encoding"""
    one_hot = np.zeros((len(labels), num_classes))
    for idx, label in enumerate(labels):
        one_hot[idx, label] = 1
    return one_hot


num_classes = 5
y_train_one_hot = one_hot_encode(y_train.values, num_classes)
y_test_one_hot = one_hot_encode(y_test.values, num_classes)

print(f"One-hot encoded train labels shape: {y_train_one_hot.shape}")
print(f"One-hot encoded test labels shape: {y_test_one_hot.shape}")

In [ ]:
# Define the RNN model
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_classes, num_layers=1):
        super(SentimentRNN, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Bidirectional LSTM instead of simple RNN for better performance
        # You can change to nn.RNN if you want a simpler RNN
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.5 if num_layers > 1 else 0
        )
        
        # Dense layers
        lstm_output_size = hidden_size * 2  # Because bidirectional
        self.fc1 = nn.Linear(lstm_output_size, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        # Embedding
        x = self.embedding(x)
        
        # LSTM
        lstm_out, (hidden, cell) = self.lstm(x)
        
        # Use the last output for classification
        last_output = lstm_out[:, -1, :]
        
        # Dense layers
        x = self.fc1(last_output)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = torch.sigmoid(x)  # Sigmoid for multi-label classification
        
        return x


# Initialize the model
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 2
hidden_size = 100
num_classes = 5

model = SentimentRNN(vocab_size, embedding_dim, hidden_size, num_classes, num_layers=1)
model = model.to(device)

print(f"Model initialized:")
print(model)

In [ ]:
# Convert data to PyTorch tensors
train_seq_tensor = torch.from_numpy(train_seq_padded).long().to(device)
y_train_tensor = torch.from_numpy(y_train_one_hot).float().to(device)

test_seq_tensor = torch.from_numpy(test_seq_padded).long().to(device)
y_test_tensor = torch.from_numpy(y_test_one_hot).float().to(device)

print(f"Train sequences tensor shape: {train_seq_tensor.shape}")
print(f"Train labels tensor shape: {y_train_tensor.shape}")
print(f"Test sequences tensor shape: {test_seq_tensor.shape}")
print(f"Test labels tensor shape: {y_test_tensor.shape}")

In [ ]:
# Create data loaders
batch_size = 32
train_dataset = TensorDataset(train_seq_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(test_seq_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

In [ ]:
# Define loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy for multi-label classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function: BCELoss")
print("Optimizer: Adam")

In [ ]:
# Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for X_batch, y_batch in tqdm(train_loader, desc="Training"):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy (exact match for all 5 classes)
        pred_classes = torch.argmax(outputs, dim=1)
        true_classes = torch.argmax(y_batch, dim=1)
        correct += (pred_classes == true_classes).sum().item()
        total += y_batch.size(0)
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy


# Validation function
def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in tqdm(val_loader, desc="Validating"):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            total_loss += loss.item()
            
            pred_classes = torch.argmax(outputs, dim=1)
            true_classes = torch.argmax(y_batch, dim=1)
            correct += (pred_classes == true_classes).sum().item()
            total += y_batch.size(0)
    
    avg_loss = total_loss / len(val_loader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy


print("Training and validation functions defined!")

In [ ]:
# Train the model
num_epochs = 10
history = {
    'train_loss': [],
    'val_loss': [],
    'train_accuracy': [],
    'val_accuracy': []
}

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

print("\nTraining completed!")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['train_accuracy'], label='Train Accuracy')
plt.plot(history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Function to predict sentiment for new text
def predict_sentiment(text, model, tokenizer, max_len, device):
    """
    Predict sentiment for given text
    Returns: sentiment_class, probabilities
    """
    sentiment_map = {
        0: 'Very Negative',
        1: 'Negative',
        2: 'Neutral',
        3: 'Positive',
        4: 'Very Positive'
    }
    
    # Tokenize and pad
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence, _ = pad_sequences_custom(sequence, maxlen=max_len, padding='post')
    
    # Convert to tensor
    input_tensor = torch.from_numpy(padded_sequence).long().to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
    
    # Get predicted class
    pred_class = torch.argmax(output, dim=1).item()
    probabilities = output.cpu().numpy()[0]
    
    return sentiment_map[pred_class], probabilities


print("Prediction function defined!")

In [ ]:
# Test the model with sample sentences
test_sentences = [
    "hate it",
    "i love this movie",
    "it is okay",
    "worst movie ever",
    "absolutely fantastic"
]

print("Testing model on sample sentences:\n")
for sentence in test_sentences:
    sentiment, probabilities = predict_sentiment(sentence, model, tokenizer, max_len, device)
    print(f"Sentence: '{sentence}'")
    print(f"Predicted Sentiment: {sentiment}")
    print(f"Probabilities: {probabilities}")
    print()

In [ ]:
# Save the model
model_save_path = 'sentiment_rnn_pytorch.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

# Save tokenizer
tokenizer_save_path = 'sentiment_tokenizer.pkl'
with open(tokenizer_save_path, 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"Tokenizer saved to {tokenizer_save_path}")

# Save configuration
import json
config = {
    'vocab_size': vocab_size,
    'embedding_dim': embedding_dim,
    'hidden_size': hidden_size,
    'num_classes': num_classes,
    'max_len': max_len,
    'sentiment_map': {0: 'Very Negative', 1: 'Negative', 2: 'Neutral', 3: 'Positive', 4: 'Very Positive'}
}
config_save_path = 'sentiment_config.json'
with open(config_save_path, 'w') as f:
    json.dump(config, f)
print(f"Config saved to {config_save_path}")